# Notebook 3: Daten Extrahieren — E in ETL

## Was bedeutet ETL?

```
E → Extract   (Extrahieren): Daten aus Quellen lesen
T → Transform (Transformieren): Daten bereinigen und umformen
L → Load      (Laden): Daten in Ziel speichern
```

Dieses Notebook behandelt den **Extract**-Schritt.

## Lernziele

Nach diesem Notebook kannst du:
- CSV-Dateien einlesen (mit verschiedenen Trennzeichen, Encodings)
- JSON-Daten verarbeiten (flach und verschachtelt)
- Excel-Dateien einlesen (einzelne und mehrere Sheets)
- Daten von einer REST-API abrufen
- Mehrere Dateien automatisiert einlesen und zusammenführen
- Häufige Probleme beim Einlesen erkennen und lösen

---

In [ ]:
import pandas as pd

# json: Python-Standardmodul zum Lesen und Schreiben von JSON-Dateien
import json

# os: Python-Standardmodul für Dateisystem-Operationen (Ordner erstellen, Pfade, ...)
import os

print("Bibliotheken geladen")


---
# 1. Beispieldaten erstellen

Wir erzeugen zunächst realistische Beispieldateien, mit denen wir arbeiten werden.

### Dateien sicher öffnen mit `with`

Beim Lesen und Schreiben von Dateien verwenden wir immer das `with`-Statement:

```python
with open("datei.csv", "w", encoding="utf-8") as f:
    f.write(inhalt)
```

**Was macht `with`?**
- Es öffnet die Datei und gibt sie unter dem Namen `f` bereit
- Nach dem eingerückten Block wird die Datei **automatisch geschlossen** — auch wenn ein Fehler auftritt
- Ohne `with` müsste man `f.close()` manuell aufrufen und könnte es vergessen

Die Parameter von `open()`:
| Parameter | Bedeutung |
|-----------|-----------|
| `"datei.csv"` | Pfad zur Datei |
| `"w"` | Modus: `"w"` = schreiben, `"r"` = lesen, `"a"` = anhängen |
| `encoding="utf-8"` | Zeichenkodierung — wichtig für Umlaute |

In [ ]:
# Verzeichnis für Beispieldaten erstellen
os.makedirs("daten", exist_ok=True)

# --- CSV-Datei 1: Standard (Komma-getrennt, englisches Format) ---
csv_standard = """id,produkt,kategorie,preis,menge,datum
1,Laptop Pro,Elektronik,999.99,2,2024-01-15
2,USB-Maus,Zubehör,29.90,5,2024-01-15
3,Monitor 27\",Elektronik,349.00,1,2024-01-16
4,Tastatur,Zubehör,49.90,3,2024-01-16
5,Webcam HD,Zubehör,79.00,2,2024-01-17
6,SSD 1TB,Speicher,89.90,4,2024-01-17
7,RAM 16GB,Speicher,65.00,3,2024-01-18
8,Headset,Zubehör,,1,2024-01-18
"""
with open("daten/verkauf_standard.csv", "w", encoding="utf-8") as f:
    f.write(csv_standard)

# --- CSV-Datei 2: Deutsches Format (Semikolon, Komma als Dezimaltrennzeichen) ---
csv_deutsch = """id;produkt;kategorie;preis;menge;datum
1;Laptop Pro;Elektronik;"999,99";2;15.01.2024
2;USB-Maus;Zubehör;"29,90";5;15.01.2024
3;Monitor;Elektronik;"349,00";1;16.01.2024
4;Tastatur;Zubehör;"49,90";3;16.01.2024
"""
with open("daten/verkauf_deutsch.csv", "w", encoding="utf-8") as f:
    f.write(csv_deutsch)

# --- JSON-Datei ---
json_daten = [
    {"kunden_id": 101, "name": "Anna Schmidt", "email": "anna@beispiel.de",
     "adresse": {"stadt": "Hamburg", "plz": "20095"}, "bestellungen": 12},
    {"kunden_id": 102, "name": "Klaus Müller", "email": "klaus@beispiel.de",
     "adresse": {"stadt": "Berlin", "plz": "10115"}, "bestellungen": 5},
    {"kunden_id": 103, "name": "Sara Wagner", "email": "sara@beispiel.de",
     "adresse": {"stadt": "München", "plz": "80331"}, "bestellungen": 23},
]
with open("daten/kunden.json", "w", encoding="utf-8") as f:
    json.dump(json_daten, f, ensure_ascii=False, indent=2)

print("Beispieldateien erstellt in ./daten/")
os.listdir("daten")

---
# 2. CSV-Dateien einlesen

CSV (Comma Separated Values) ist das häufigste Format für Datenaustausch.

`pd.read_csv()` ist die wichtigste Funktion — mit vielen nützlichen Parametern.

In [ ]:
# Einfachste Variante: Standard-CSV
df = pd.read_csv("daten/verkauf_standard.csv")
df

In [ ]:
# Datentypen prüfen - was hat pandas automatisch erkannt?
df.dtypes

In [ ]:
# Problem: 'datum' wurde als Text erkannt, nicht als Datum
# Lösung: parse_dates Parameter
df = pd.read_csv("daten/verkauf_standard.csv", parse_dates=["datum"])
print(df.dtypes)
df

In [ ]:
# Wichtige pd.read_csv() Parameter:
df_mit_optionen = pd.read_csv(
    "daten/verkauf_standard.csv",
    parse_dates=["datum"],       # Spalten als Datum parsen
    index_col="id",              # Diese Spalte als Index verwenden
    usecols=["id", "produkt",    # Nur diese Spalten laden (spart Speicher!)
             "preis", "menge",
             "datum"],
)
df_mit_optionen

## 2.1 Das deutsche CSV-Format

In Deutschland verwenden viele Programme (Excel, SAP) das Semikolon als Trennzeichen und das Komma als Dezimaltrennzeichen. Das muss explizit angegeben werden!

In [ ]:
# FALSCH: Standard-Einstellungen auf deutschen CSV
df_falsch = pd.read_csv("daten/verkauf_deutsch.csv")
print("Falsch eingelesen:")
display(df_falsch)
print(df_falsch.dtypes)

In [ ]:
# RICHTIG: Deutsche Einstellungen
df_deutsch = pd.read_csv(
    "daten/verkauf_deutsch.csv",
    sep=";",           # Semikolon als Trennzeichen
    decimal=",",       # Komma als Dezimaltrenner
    encoding="utf-8",  # Zeichenkodierung (für Umlaute)
)
print("Richtig eingelesen:")
display(df_deutsch)
print(df_deutsch.dtypes)

## 2.2 Häufige Probleme beim CSV-Import

Eine kleine Diagnose-Routine hilft beim Einstieg in einen neuen Datensatz:

In [ ]:
def csv_diagnostik(dateipfad):
    """Zeigt eine Diagnose-Übersicht für eine CSV-Datei."""
    df = pd.read_csv(dateipfad)
    
    print(f"=== Diagnostik: {dateipfad} ===")
    print(f"Zeilen: {df.shape[0]:,}  |  Spalten: {df.shape[1]}")
    print(f"Speicher: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
    print()
    print("Spaltenuebersicht:")
    for col in df.columns:
        n_fehlend = df[col].isnull().sum()
        fehlend_pct = 100 * n_fehlend / len(df)
        print(f"  {col:<20} {str(df[col].dtype):<12} "
              f"fehlend: {n_fehlend} ({fehlend_pct:.0f}%)")
    print()
    return df

df = csv_diagnostik("daten/verkauf_standard.csv")

---
# 3. JSON-Dateien einlesen

JSON (JavaScript Object Notation) ist das Standardformat für APIs und Web-Daten.

In [ ]:
# Variante A: Direkt mit pandas (einfachste Methode, wenn Struktur flach ist)
df_json = pd.read_json("daten/kunden.json")
df_json

In [ ]:
# Problem: 'adresse' ist ein verschachteltes Dictionary in einer Spalte!
print(df_json["adresse"][0])  # Das ist noch ein Dict, keine sauberen Spalten

In [ ]:
# Variante B: JSON manuell laden und json_normalize für verschachtelte Daten
with open("daten/kunden.json", "r", encoding="utf-8") as f:
    kunden_daten = json.load(f)

# json_normalize "plattet" verschachtelte Strukturen
df_kunden = pd.json_normalize(
    kunden_daten,
    sep="_"   # Trennzeichen für verschachtelte Schlüssel: adresse.stadt -> adresse_stadt
)
df_kunden

---
### Übung 1: CSV einlesen und erkunden

Erstelle eine eigene CSV-Datei `daten/meine_daten.csv` mit mindestens 5 Zeilen und 4 Spalten deiner Wahl (z.B. eine fiktive Produktliste, Kundenliste, etc.).

Dann:
1. Lese die Datei mit `pd.read_csv()` ein
2. Führe `csv_diagnostik()` darauf aus
3. Zeige die ersten und letzten 3 Zeilen
4. Berechne eine einfache Statistik (z.B. Summe oder Durchschnitt einer Zahlspalte)

> **Übung → siehe [`03_ETL_Extract_Aufgaben.ipynb`](03_ETL_Extract_Aufgaben.ipynb)**

---
# 4. Daten von einer REST-API abrufen

APIs (Application Programming Interfaces) liefern Daten über HTTP-Anfragen — meist im JSON-Format.

Wir verwenden die öffentliche JSONPlaceholder-API als Beispiel.

In [ ]:
import requests  # HTTP-Bibliothek

# GET-Anfrage an eine öffentliche Test-API
url = "https://jsonplaceholder.typicode.com/posts"

antwort = requests.get(url, timeout=10)

# Status-Code prüfen: 200 = OK, 404 = nicht gefunden, 500 = Serverfehler
print(f"Status-Code: {antwort.status_code}")
antwort.raise_for_status()  # Wirft Fehler bei HTTP-Fehlern

# JSON-Antwort parsen
daten = antwort.json()
print(f"Anzahl Datensätze: {len(daten)}")
print(f"Erster Datensatz: {daten[0]}")

In [ ]:
# API-Daten in DataFrame umwandeln (wenn Daten vorhanden)
if daten:
    df_api = pd.DataFrame(daten)
    print(f"Shape: {df_api.shape}")
    display(df_api.head())
else:
    print("Keine Daten geladen (kein Internet oder API nicht erreichbar)")

In [ ]:
# Wiederverwendbare Funktion zum API-Abruf
def api_daten_laden(url, params=None):
    """Lädt JSON-Daten von einer REST-API und gibt einen DataFrame zurück."""
    antwort = requests.get(url, params=params, timeout=10)
    antwort.raise_for_status()
    daten = antwort.json()
    
    # Manche APIs geben ein Dict zurück, andere eine Liste
    if isinstance(daten, list):
        return pd.DataFrame(daten)
    elif isinstance(daten, dict):
        return pd.json_normalize(daten)

# Beispiel: Nur die ersten 5 Posts laden
df_posts = api_daten_laden(
    "https://jsonplaceholder.typicode.com/posts",
    params={"_limit": 5}
)
df_posts

---
# 5. Excel-Dateien einlesen

In [ ]:
# Excel-Datei mit mehreren Blättern erstellen
with pd.ExcelWriter("daten/bericht.xlsx", engine="openpyxl") as writer:
    pd.DataFrame([
        {"monat": "Januar", "umsatz": 45000, "kosten": 32000},
        {"monat": "Februar", "umsatz": 51000, "kosten": 33500},
        {"monat": "März", "umsatz": 49000, "kosten": 31000},
    ]).to_excel(writer, sheet_name="Q1", index=False)

    pd.DataFrame([
        {"monat": "April", "umsatz": 55000, "kosten": 34000},
        {"monat": "Mai", "umsatz": 62000, "kosten": 36000},
        {"monat": "Juni", "umsatz": 58000, "kosten": 35500},
    ]).to_excel(writer, sheet_name="Q2", index=False)

print("Excel-Datei erstellt: daten/bericht.xlsx")

# Einlesen mit pd.read_excel()
df_q1 = pd.read_excel("daten/bericht.xlsx", sheet_name="Q1")
df_q2 = pd.read_excel("daten/bericht.xlsx", sheet_name="Q2")

print("\nQ1-Daten:")
display(df_q1)

print("\nAlle Sheets einlesen (als Dict):")
alle_sheets = pd.read_excel("daten/bericht.xlsx", sheet_name=None)  # None = alle Sheets
for sheet_name, df_sheet in alle_sheets.items():
    print(f"  Sheet '{sheet_name}': {df_sheet.shape}")

---
# 6. Mehrere Dateien automatisiert einlesen

In der Praxis gibt es oft viele gleichartige Dateien (z.B. monatliche Exporte), die zusammengeführt werden sollen.

In [ ]:
# Mehrere CSV-Dateien erzeugen (simuliert monatliche Exporte)
os.makedirs("daten/monatlich", exist_ok=True)

for monat, daten in [
    ("2024-01", [[1,"Laptop",999],[2,"Maus",29]]),
    ("2024-02", [[3,"Monitor",349],[4,"Tastatur",49]]),
    ("2024-03", [[5,"Webcam",79],[6,"Headset",89]]),
]:
    pd.DataFrame(daten, columns=["id","produkt","preis"]).to_csv(
        f"daten/monatlich/verkauf_{monat}.csv", index=False
    )

print("Monatsdateien erstellt:")
print(os.listdir("daten/monatlich"))

Alle CSV-Dateien im Verzeichnis werden automatisch eingelesen. `os.listdir()` listet alle Dateien auf — eine List Comprehension filtert davon nur die `.csv`-Dateien heraus.

In [ ]:
dateipfade = [
    os.path.join("daten/monatlich", f)
    for f in sorted(os.listdir("daten/monatlich"))
    if f.endswith(".csv")
]
print(f"Gefundene Dateien: {dateipfade}")

# Jede Datei einlesen, Dateiname als Spalte hinzufügen
df_teile = []
for pfad in dateipfade:
    df_teil = pd.read_csv(pfad)
    dateiname = os.path.basename(pfad).replace(".csv", "")
    df_teil["quelle"] = dateiname
    df_teile.append(df_teil)

# Alle Teile zu einem DataFrame zusammenfügen
df_gesamt = pd.concat(df_teile, ignore_index=True)
print(f"\nGesamter DataFrame: {df_gesamt.shape}")
df_gesamt

---
### Übung 2: API und Daten zusammenführen

Nutze die öffentliche API `https://jsonplaceholder.typicode.com/users` (liefert fiktive User-Daten).

1. Lade die Daten mit `api_daten_laden()`
2. Zeige die verfügbaren Spalten
3. Erstelle einen reduzierten DataFrame mit nur: `id`, `name`, `email`, `phone`
4. Zeige alle User, deren Name mit "L" anfängt

Falls kein Internet: Erstelle die Daten manuell als Liste von Dictionaries.

> **Übung → siehe [`03_ETL_Extract_Aufgaben.ipynb`](03_ETL_Extract_Aufgaben.ipynb)**

---
## Zusammenfassung

In diesem Notebook hast du gelernt, Daten aus verschiedenen Quellen einzulesen:

| Quelle | Funktion / Methode |
|--------|-------------------|
| **CSV (Standard)** | `pd.read_csv()` — mit `parse_dates`, `usecols`, `index_col` |
| **CSV (Deutsch)** | `sep=";"`, `decimal=","`, `encoding="utf-8"` |
| **Excel** | `pd.read_excel()` — einzelne und mehrere Sheets |
| **JSON** | `pd.read_json()`, `json.load()`, `pd.json_normalize()` |
| **REST-API** | `requests.get()`, JSON-Antwort in DataFrame umwandeln |
| **Mehrere Dateien** | `os.listdir()` + Schleife + `pd.concat()` |

Außerdem: `with open(...)` für sicheres Lesen und Schreiben von Dateien.

---
# Weitere Übungsaufgaben

Die folgenden Aufgaben beziehen sich auf die Themen dieses Notebooks. Musterlösungen findest du im Ordner `Loesungen/`.

### Aufgabe 1 — Deutsches CSV einlesen

Lies die folgende semikolon-getrennte CSV-Datei mit deutschem Dezimaltrenner ein.
Führe danach eine Diagnose durch: Shape, Datentypen, fehlende Werte.

> **Lösung → [`03_ETL_Extract_Aufgaben.ipynb`](03_ETL_Extract_Aufgaben.ipynb)**

### Aufgabe 2 — Mehrere Dateien zusammenführen

Erstelle drei CSV-Dateien für drei Filialen und lies sie automatisiert ein.
Füge beim Einlesen jeweils den Dateinamen als Spalte `filiale` hinzu.
Berechne abschließend den Gesamtumsatz pro Filiale.

> **Lösung → [`03_ETL_Extract_Aufgaben.ipynb`](03_ETL_Extract_Aufgaben.ipynb)**

### Aufgabe 3 — Verschachteltes JSON flachklopfen

Lies die folgende JSON-Struktur ein und erzeuge einen flachen DataFrame mit den Spalten: `id`, `titel`, `autor_name`, `autor_email`, `erscheinungsjahr`.

> **Lösung → [`03_ETL_Extract_Aufgaben.ipynb`](03_ETL_Extract_Aufgaben.ipynb)**